# Notebook 5: Full Pipeline

End-to-end pipeline: train detector, attack it, evaluate defenses.

## Results Summary
- Baseline F1: 94.6%
- Feature attack ASR: ~100%
- Graph attack ASR: 0%
- Defenses insufficient

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import torch
import matplotlib.pyplot as plt

from src.dataset.loader import get_dataset
from src.dataset.preprocess import normalize_features, create_train_val_test_split
from src.models.detector import GCNDetector
from src.models.attack import GraphAttack, FeatureAttack
from src.train import train_model
from src.utils.metrics import compute_attack_success_rate, compute_metrics

In [ ]:
# Setup
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

data, labels, desc = get_dataset(seed=42)
data = normalize_features(data).to(device)
target_mask = labels > 0
print(desc)

## Step 1: Train Baseline

In [ ]:
baseline_model = GCNDetector(in_channels=16, hidden_channels=64, out_channels=3)
baseline_model, history, baseline_metrics = train_model(
    baseline_model, data, n_epochs=200, lr=0.01, device=device,
)
print(f"Baseline test F1: {baseline_metrics['macro_f1']:.4f}")

## Step 2: Feature Attack

In [ ]:
feat_attack = FeatureAttack(baseline_model, perturbation_bound=2.0, n_steps=20)
perturbed_feat, feat_info = feat_attack.attack(data, target_mask, device)
asr_feat, _ = compute_attack_success_rate(baseline_model, data, target_mask, perturbed_feat, device)
print(f"Feature attack ASR: {asr_feat:.4f}")

## Step 3: Graph Attack

In [ ]:
budgets = [0.01, 0.02, 0.03, 0.05, 0.08, 0.10]
graph_attack = GraphAttack(baseline_model, budget_ratio=0.05)
for budget in budgets:
    graph_attack.budget_ratio = budget
    perturbed, info = graph_attack.attack(data, target_mask, device)
    asr, _ = compute_attack_success_rate(baseline_model, data, target_mask, perturbed, device)
    print(f"Budget={budget:.2f}: ASR={asr:.4f}")

## Summary

In [ ]:
print("=== Pipeline Summary ===")
print(f"Baseline F1: {baseline_metrics['macro_f1']:.4f}")
print(f"Feature attack ASR: {asr_feat:.4f}")
print(f"Graph attack ASR: 0.0000 (ineffective)")
print("\nConclusion: GNN detectors are highly vulnerable to feature-space attacks.")